In [7]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [8]:
from litellm import completion
import os
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
# The Simplest LiteLLM Example — Unified API

from litellm import completion

question = "What is RAG? Explain in two lines"

response = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": question
        }
    ]
)

print("Groq : ",response.choices[0].message.content)

Groq :  RAG (Retrieval‑Augmented Generation) is an NLP framework that combines a language model with a document retriever, feeding relevant external text into the model’s input.  
This hybrid approach lets the model generate responses that are grounded in up‑to‑date, domain‑specific knowledge beyond its static training data.


In [10]:
from litellm import completion

prompt = "Explain RAG in one sentence."

providers = [
    {"Groq": "groq/openai/gpt-oss-20b"},
    {"Gemini": "gemini/gemini-3.8-flash"}
]

for provider in providers:
    for label, model in provider.items():
        try:
            r = completion(
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            print(f"{label:<15}: {r.choices[0].message.content}")

        except Exception as e:
            print(f"{label:<15}: {type(e).__name__}")

Groq           : RAG (Retrieval‑Augmented Generation) is a technique that boosts a language model’s outputs by first fetching relevant documents from an external knowledge base and then conditioning the generation on that retrieved content.
Gemini         : **Retrieval-Augmented Generation (RAG)** is a technique that improves AI responses by fetching relevant facts from an external knowledge base before the language model generates an answer, ensuring outputs are accurate and up-to-date.


In [11]:
# Automatic Fallbacks — When Onekey Goes Down

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is an LLM Gateway? Explain in short"}],
    fallbacks=[
        "groq/openai/gpt-oss-20b",
        "gemini/gemini-3.8-flash"
    ]
)

print("Response:", response.choices[0].message.content, "...")
print("\nAnswered Model", response.model)



13:19:14 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gpt-4o-mini: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.
Traceback (most recent call last):
  File "/home/amir/llm-production-engineering/.venv/lib/python3.14/site-packages/litellm/llms/openai/openai.py", line 921, in acompletion
    openai_aclient: AsyncOpenAI = self._get_openai_client(
                                  ~~~~~~~~~~~~~~~~~~~~~~~^
        is_async=True,
        ^^^^^^^^^^^^^^
    ...<7 lines>...
        shared_session=shared_session,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/amir/llm-production-engineering/.venv/lib/python3.14/site-packages/litellm/llms/openai/openai.py", line 415, in _get_openai_client
    else AsyncOpenAI(
         ~~~~~~~~~~~^
        api_key=api_key,
        ^^^^^^^^^^


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



Task was destroyed but it is pending!
task: <Task pending name='Task-46' coro=<LoggingWorker._worker_loop() running at /home/amir/llm-production-engineering/.venv/lib/python3.14/site-packages/litellm/litellm_core_utils/logging_worker.py:219>>


Response: **LLM Gateway (Large Language Model Gateway)**  

An LLM Gateway is a middleware layer that sits between your applications (or end‑users) and one or more large language model services (e.g., OpenAI, Anthropic, Azure GPT). It handles:

1. **Routing** – sends requests to the appropriate LLM or model variant.
2. **Authentication & Authorization** – verifies who can use the model and enforces access controls.
3. **Rate‑limiting & Quotas** – protects your backend from overload and keeps costs in check.
4. **Context Management** – maintains conversation history or state across requests.
5. **Logging & Analytics** – tracks usage, performance, and billing.
6. **Security & Compliance** – masks sensitive data, ensures data residency, and applies privacy policies.

In short, an LLM Gateway abstracts the complexities of calling LLM APIs, giving you a unified, secure, and manageable interface to leverage powerful language models in your own systems. ...

Answered Model openai/gpt-oss-20b


In [17]:
# Cost Tracking — Know Where Your Money Goes
from litellm import completion_cost

response = completion(
    model="gemini/gemini-3.8-flash",
    messages=[{"role": "user", "content": "What is llm evaluation. Explain in short"}]
)

cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")


Response:     **LLM evaluation** is the process of measuring the performance, accuracy, safety, and quality of a Large Language Model’s outputs. 

In short, it answers: *“Is the model giving reliable, safe, and useful answers?”*

### What it evaluates:
* **Accuracy & Factuality:** Does it give the right information, or does it hallucinate (make things up)?
* **Relevance & Coherence:** Does it actually answer the user's prompt in clear language?
* **Safety & Bias:** Does it generate toxic, biased, or harmful content?
* **Task Performance:** How well does it handle specific tasks like coding, summarizing, or math?

### How it is done:
1. **Automated Benchmarks:** Standardized tests (e.g., MMLU, GSM8K) with objective scoring.
2. **Human Evaluation:** Real people reviewing and rating the responses.
3. **LLM-as-a-Judge:** Using a powerful model (like GPT-4) to grade the outputs of another model.

Input tokens:  10
Output tokens: 619
Cost:         $0.00232875


In [18]:
# Caching — Don't Pay Twice for the Same Question
import litellm

litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []
litellm.cache = None

In [19]:
import time
import litellm
from litellm import completion
from litellm.caching import Cache

litellm.cache = Cache(type="local")

prompt = "What is llm gateway explain in short"

start = time.time()

r1 = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    caching=True
)

t1 = time.time() - start

print(f"First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

start = time.time()

r2 = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    caching=True
)

t2 = time.time() - start

print(f"Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\nSpeedup: {t1 / t2:.1f}x faster than first call")

First call (API):   0.80s — **LLM Gateway (in a nutshell)**

An LLM gateway is a middleware layer that sits between users/clients and large‑language‑model services (e.g., OpenAI, Azure GPT, AWS Bedrock). It handles:

- **Authentication & authorization** – ensures only permitted users can call the model.  
- **Request routing & load balancing** – distributes calls across available instances or providers.  
- **Caching & rate‑limiting** – reduces cost and prevents over‑usage.  
- **Input/output formatting** – normalizes requests and responses for downstream systems.  
- **Monitoring & logging** – tracks usage, latency, and errors for billing and debugging.

In short, it’s the “traffic cop” that makes interacting with an LLM reliable, secure, and efficient.
Second call (cache): 0.0916s — **LLM Gateway (in a nutshell)**

An LLM gateway is a middleware layer that sits between users/clients and large‑language‑model services (e.g., OpenAI, Azure GPT, AWS Bedrock). It handles:

- **Authenticat

Observability — Log Every Single Call

In [23]:
import json
import litellm
from litellm import completion

call_logs = []


def log_success(kwargs, completion_response, start_time, end_time):
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round(
            (end_time - start_time).total_seconds(), 2
        ),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })


def log_failure(kwargs, completion_response, start_time, end_time):
    print("Call failed:", kwargs.get("exception"))


litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]


for q, user in [
    ("What is RAG?", "Amir"),
    ("Explain transformers.", "student"),
    ("What is fine-tuning?", "Amir"),
]:
    completion(
        model="groq/openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": q
            }
        ],
        user=user
    )


print(json.dumps(call_logs, indent=2, default=str))

[
  {
    "model": "openai/gpt-oss-20b",
    "prompt": "What is RAG?",
    "input_tokens": 76,
    "output_tokens": 601,
    "latency_sec": 1.15,
    "cost_usd": 0.000186,
    "user": "Amir"
  },
  {
    "model": "openai/gpt-oss-20b",
    "prompt": "Explain transformers.",
    "input_tokens": 74,
    "output_tokens": 2048,
    "latency_sec": 2.82,
    "cost_usd": 0.00061995,
    "user": "student"
  }
]


Integrating the Gateway with LangChain

In [24]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatLiteLLM(
    model="ollama/qwen3:8b",
    api_base="http://127.0.0.1:11434",
    temperature=0.3
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named mero-ai. Be concise."),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "question": "What is an LLM Gateway in 3 bullets?"
})

print(answer)

- **Intermediary Layer**: Acts as a bridge between users and large language models (LLMs), managing requests and responses.  
- **Security & Control**: Enforces authentication, rate limiting, and access policies to ensure safe and efficient usage.  
- **Optimization**: Enhances performance through caching, model selection, and response formatting for diverse use cases.


In [26]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Primary model
primary = ChatLiteLLM(model="gemini/gemini-3.8-flash")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="groq/openai/gpt-oss-20b", temperature=0.2)
fallback_2 = ChatLiteLLM(model="ollama/qwen3:8b",api_base = "http://127.0.0.1:11434", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Evaluation?"})
print(result)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Call failed: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 13h22m8.962614848s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        